In [16]:
import pandas as pd
import numpy as np
import config
import torch
import torch.nn as nn
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
def split_by_machine(df, train_ratio=0.8):
    train_list, test_list = [], []
    for id, data in df.groupby('machineID'):
        data = data.sort_values('datetime')
        split_idx = int(len(data) * train_ratio)
        train_list.append(data.iloc[:split_idx])
        test_list.append(data.iloc[split_idx:])
    return pd.concat(train_list), pd.concat(test_list)

In [3]:
# 加载数据集
errors = pd.read_csv(config.RAW_DATA_DIR / "PdM_errors.csv")
failures = pd.read_csv(config.RAW_DATA_DIR / "PdM_failures.csv")
machines = pd.read_csv(config.RAW_DATA_DIR / "PdM_machines.csv")
maint = pd.read_csv(config.RAW_DATA_DIR / "PdM_maint.csv")
telemetry = pd.read_csv(config.RAW_DATA_DIR / "PdM_telemetry.csv")

In [4]:
# 日期格式
date_tables = [errors, failures, maint, telemetry]
for df in date_tables:
    df['datetime'] = pd.to_datetime(df['datetime'], format="%Y-%m-%d %H:%M:%S")
    df.sort_values(['machineID', 'datetime'], inplace=True, ignore_index=True)

In [5]:
# 合并数据集——machines
dataset = pd.merge(left=machines, right=telemetry,
                    on=['machineID']).sort_values(['datetime', 'machineID'])

# 合并数据集——errors
errors.sort_values(['datetime', 'machineID'], inplace=True)
dataset = pd.merge(left=dataset, right=errors,
                    how='left',
                    on=['datetime', 'machineID'])

# 合并数据集——maint
maint.sort_values(['datetime','machineID'], inplace=True)
dataset = pd.merge(left=dataset, right=maint,
                    how='left',
                    on=['datetime', 'machineID'])

# 合并数据集——failures（计算RUL）
failures['RUL'] = failures['datetime']
failures.sort_values(['datetime', 'machineID'], inplace=True)
dataset = pd.merge_asof(left=dataset, right=failures[['datetime', 'machineID', 'RUL']],
                        on='datetime',
                        by='machineID',
                        direction='forward').sort_values(['machineID','datetime']).reset_index(drop=True)
dataset['RUL'] = (dataset['RUL'] - dataset['datetime']).dt.total_seconds() / (24 * 3600)

In [6]:
# 填补数据集
dataset['RUL'] = dataset['RUL'].fillna(365)

In [7]:
train, test = split_by_machine(dataset, train_ratio=0.5)

# 特征预处理
nomodify = ['machineID', 'datetime', 'RUL', 'age']
label = ['model']
numerical_features = [col for col in train.select_dtypes(include='float64').columns
                      if col not in nomodify + label]
categorical_features = [col for col in train.select_dtypes(include='str').columns
                        if col not in nomodify + label]

preprocessor = ColumnTransformer(
    transformers=[
        ("", "passthrough", nomodify),
        ("label", OneHotEncoder(drop="first"), label),
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown='ignore'), categorical_features)
        ]
    )

preprocessor.fit(train)
train = pd.DataFrame(preprocessor.transform(train), columns=preprocessor.get_feature_names_out())
test = pd.DataFrame(preprocessor.transform(test), columns=preprocessor.get_feature_names_out())

In [8]:
# 将预处理后的2D数据重塑为3D数据
def create_rolling_window(data, seq_len=7):
    """
    输入
    data: Dataframe, 包含特征列+machineID+datetime+RUL
    seq_len: int, 序列长度(时间长度)
    输出
    input: np.ndarray shape:[batch, seq_len, feature_num]
        batch: 总的批数
        seq_len: 时间窗口长度
        feature_num: 特征维度
    target: np.ndarray shape:[batch, 1]
        每批数据对应的RUL
    """
    input, target = [], []
    for machine_id in data['__machineID'].unique():
        machine_data = data[data['__machineID'] == machine_id].sort_values(['__machineID'])

        feature_cols = [col for col in machine_data.columns if col not in ['__machineID', '__datetime', '__RUL']]
        features = machine_data[feature_cols].values

        rul = machine_data['__RUL'].values

        for i in range(len(machine_data) - seq_len + 1):
            input.append(features[i:i+seq_len])
            target.append(rul[i+seq_len-1])

    return np.array(input, dtype=float), np.array(target, dtype=float).reshape(-1, 1)

train_input, train_target = create_rolling_window(train, seq_len=7)
test_input, test_target = create_rolling_window(test, seq_len=7)

In [ ]:
train_dataset = TensorDataset(torch.tensor(train_input), torch.tensor(train_target))
test_dataset = TensorDataset(torch.tensor(test_input), torch.tensor(test_target))

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [13]:
def train_one_epoch(model, datalodaer, loss_function, optimizer, device='cpu'):
    total_loss = 0
    total_samples = 0

    model.train()

    for input, target in tqdm(datalodaer, desc='train'):
        input, target = input.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(input)
        loss = loss_function(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * input.shape[0]
        total_samples += target.shape[0]

    return total_loss / total_samples

In [17]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            batch_first=True
            )
        self.linear = nn.Linear(hidden_size, 1)

    def forward(self, x):
        lstm_out, (hidden, cell) = self.lstm(x)
        last_output = lstm_out[:, -1, :]
        output = self.linear(last_output)
        return output

In [ ]:
def train():
    # 1.设备
    device = torch.device('cpu')

    # 2.数据
    dataloader = DataLoader(train_dataset, batch_size=32, shuffle=False)

    # 4.模型
    model = nn.LSTMModel(
        input_size=
        ).to(device)

    # 5.损失函数
    loss_function = nn.MSELoss()

    # 6.优化器
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

    best_loss = float('inf')
    for epoch in range(1, config.EPOCHS + 1):
        print(f'============= EPOCH {epoch} ===========')
        loss = train_one_epoch(model, dataloader, loss_function, optimizer, device)
        print(f'loss: {loss}')

        # 保存模型
        if loss < best_loss:
            best_loss = loss
            torch.save(model.state_dict(), config.MODELS_DIR / 'best.pth')
            print('保存模型')